### Install Libraries

In [13]:
# %pip install ddgs trafilatura
%pip install openai-agents

Note: you may need to restart the kernel to use updated packages.


### Setup

In [14]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

MODEL="gpt-4.1-mini"

### Step 1: Define the Tools

In [15]:
@function_tool # Instructs OpenAI SDK to set these up as tools for agents
def search_web(query: str):
    """Search the web using DuckDuckGo Browser. Returns 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2)

In [16]:
#'beautiful soup' is another package you can use if you need more control
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

### Step 2: The Agents

The Agent Prompts tell the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce



#### Research Agent

In [17]:

RESEARCH_AGENT_PROMPT="""You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching. ALLOW yourself moments to simply think on what has already been gathered to determine the best next step.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

*** IMPORTANT:
After each search with search_web, you MUST first explain your reasoning:
- Which URLs seem most relevant and why?
- Which ones you will fetch and why?
- Which ones you will discard and why?
Only after writing out your reasoning should you call fetch_url.
***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs. If a URL doesn't provide useful information, discard it but note why.
4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

#### Orchestrator Agent

In [18]:
# My original prompt
# ORCHESTRATOR_AGENT_PROMPT = """
# You are an orchestrator for multi agent researchers. Your job is to take in various research briefs and select the best one for the publication of an article.

# You MUST call for at least two completed research briefs. 

# You have access to a tool:
# - research_agent: Create a research brief based on a query

# Your typical process:
# 1. Pass your query to your research agent and obtain a research brief
# 2. Reflect on the research results — which sources look most relevant and why?
# 3. Call your research agent again for a second brief
# 4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
# 5. If you feel the research isn't enough or too similar, call your research agent again to create additional briefs.
# 6. When you have enough information, decide on the best research brief to return.

# Good research brief MUST include:
# - Key facts and statistics
# - Main themes and arguments from the sources
# - Notable data points
# - Source URLs for attribution

# Until you are ready, just keep working — reflecting as much as you need to.
# Do not rush.
# """
ORCHESTRATOR_AGENT_PROMPT = """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs."""

orchestrator_agent = Agent(
    name="Ochestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", ## reasoning model
    tools=[research_agent.as_tool(
        tool_name="research_agent",
        tool_description="Research a topic and return a brief with key facts, statistic themes, and source URLs. Pass topic as input",
        max_turns=20 # max turns are not inherited by the orchestrator agent and defaults to 10
    )]
)

### Let's Run it!

In [19]:
from agents import trace

with trace("Article Writer", group_id="Learning AI Engineering", metadata={"userID": "user123"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "Research the following topic and produce a comprehensive research brief: How will AI be used in Healthcare in 2030-2035",
        max_turns=30
    )

  ✅ search_web: Got results for AI in healthcare 2030 2035 future trends
  
 ✅ fetch_url: Got 1457 chars from https://www.deloitte.com/global/en/Industries/life-sciences-
  
 ✅ fetch_url: Got 23576 chars from https://www.spaceo.ai/ai-in-healthcare/future/
  ✅ search_web: Got results for AI in healthcare future 2030-2035 reports predictions
  
 ✅ fetch_url: Got 1331 chars from https://japmi.org/index.php/japmi/article/view/23
  
 ✅ fetch_url: Got 1331 chars from https://japmi.org/index.php/japmi/article/view/23
  ✅ search_web: Got results for AI healthcare workforce impact 2030-2035 advanced AI technologies explainable AI
  ❌ Failed to fetch or extract text from https://www.sciencedirect.com/science/article/pii/S0045790624002982
  ❌ Failed to fetch or extract text from https://www.sciencedirect.com/science/article/abs/pii/S0140673626006938
  ❌ Failed to fetch or extract text from https://www.mckinsey.com/industries/healthcare/our-insights/the-ai-powered-future-of-care
  ✅ search_web: Go

In [11]:
print(f"Agent: {result.last_agent.name}")
print(f"---")
display(Markdown(result.final_output))

Agent: Ochestrator Agent
---


I’ve selected the second research brief as the most comprehensive. Here it is in full:

AI in Healthcare 2030–2035: Research Brief

1. Key Developments and Trends  
• Transition from pilot projects to value-driven deployments: operational efficiency, administrative burden reduction, clinical decision support  
• Generative AI and ambient assistants for tasks like note-taking and documentation driving rapid clinician adoption  
• Core use cases: predictive analytics, workflow automation, patient engagement chatbots, diagnostic imaging analysis  
• Global clinician adoption: nearly 50% using AI by 2025; some U.S. specialties exceed 60%  
• Sustained investment: healthcare AI market revenues growing strongly through 2030  
• Persistent challenges: data infrastructure needs, privacy, clinical validation, transparency, and evolving regulatory frameworks  

2. Governance and Regulatory Landscape  
• Risk-based frameworks mandate clinical validation, fairness audits, transparency (model cards), and human oversight for high-risk AI  
• U.S.: FDA’s Pre-certification and Predetermined Change Control Plans enable adaptive regulation  
• EU: AI Act classifies medical AI as “high-risk” with obligations for human oversight, post-market monitoring, and conformity assessment  
• U.K.: Innovation-friendly post-Brexit regulation balancing patient safety and rapid adoption  
• Organizational governance: “Three Lines of Defense” model—operational management, ethics/oversight committees, independent assurance  
• Liability: product liability for developers, malpractice for clinicians, corporate negligence for institutions  
• International harmonization efforts (IMDRF, WHO, OECD) aim to align standards but full global convergence remains ongoing  

3. Notable Statistics  
• ~50% of clinicians worldwide used AI in daily practice by 2025  
• U.S. radiology and emergency medicine specialties report >60% AI adoption  
• AI-driven tools can boost healthcare provider productivity by up to 40% and patient satisfaction by up to 60%  
• AI diagnostic systems for diabetic retinopathy achieve >90% sensitivity  
• Venture investment in digital health AI has grown dramatically since 2015, sustaining capital flow into mid-2020s  

4. Illustrative Case Studies  
• Inferscience HCC Assistant: NLP improves risk-adjustment accuracy by 35%, reducing documentation burden and optimizing Medicare reimbursement  
• Moorfields Eye Hospital: AI for retinal image analysis enhances early detection of diabetic retinopathy  
• HCA Healthcare & Duke Health: AI for workflow optimization, predictive analytics, and decision support reduces bottlenecks and improves outcomes  
• Johns Hopkins Medicine & Boston Children’s: AI tailored to individual and pediatric outcomes, raising diagnostic accuracy and treatment personalization  
• Safoni: AI-driven drug discovery accelerates target identification and improves clinical trial recruitment  
• University Hospitals: NLP for population health management enables targeted interventions and better Medicare reimbursements  
• University of Florida Health: Clinical decision support systems enhance guideline adherence and treatment decisions  

Sources  
• Deloitte Life Sciences & Healthcare Predictions 2030: https://www.deloitte.com/global/en/Industries/life-sciences-health-care/collections/life-sciences-and-health-care-predictions.html  
• Statista AI in Healthcare Overview: https://www.statista.com/topics/10011/ai-in-healthcare/  
• Inferscience Case Study & AI Healthcare Case Studies: https://www.inferscience.com/7-ai-in-healthcare-case-studies-transforming-patient-care  
• Designveloper AI Healthcare Case Studies: https://www.designveloper.com/blog/case-studies-of-ai-in-healthcare/  
• Scholarly reviews and policy analyses on AI governance and regulation in healthcare  

Summary  
By 2030–2035, AI will be deeply embedded across clinical, operational, and administrative domains of healthcare. Generative and ambient AI tools will drive efficiency and reduce clinician burnout. Robust governance and risk-based regulation will ensure patient safety, fairness, and transparency. International efforts will progressively harmonize standards, while case studies demonstrate tangible gains in diagnostics, workflow, personalized treatment, and drug discovery.

### Trace logs

https://platform.openai.com/home -> logs -> traces -> Agent workflow